In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=aHETcAIh6Qm5329Uyc5s3EMU85WhlV&access_type=offline&code_challenge=ZwBlS90doqQwgcHku6TkpHSgAiMbWLdYzc6GdLs62Nw&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [2]:
!gcloud auth login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=ndHvH0qgnUz5umebCqgVltu2H7LiNx&access_type=offline&code_challenge=WRqdm8PUcwrhRUHqS4cN3hNCysWYVFph9H6slIhBIfo&code_challenge_method=S256


You are now logged in as [yt4@sanger.ac.uk].
Your current project is [open-targets-genetics-dev].  You can change this setting by running:
  $ gcloud config set project PROJECT_ID


In [3]:
import os

import hail as hl
import numpy as np
import pandas as pd
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

Loading BokehJS ...

In [4]:
"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "1g",
        "spark.driver.maxResultSize":"4g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()

session= GentropySession()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/17 13:40:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/17 13:40:40 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [7]:
path_to_release_folder="gs://open-targets-data-releases/25.03/"
#path_to_release_folder="gs://open-targets-pre-data-releases/24.12-uo_test-3/output/genetics/parquet/"
#path_to_release_folder="gs://ot_orchestration/releases/25.02_freeze1/"

In [8]:
si=StudyIndex.from_parquet(session,path_to_release_folder+"output/study/")
sl=StudyLocus.from_parquet(session,path_to_release_folder+"output/credible_set/")

# Mapping

In [5]:
si=si.df.filter(f.col("studyType")=="gwas").cache()

25/04/03 16:41:58 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [6]:
si.count()

96404

In [7]:
si.groupBy("projectId").count().show()

+-----------+-----+
|  projectId|count|
+-----------+-----+
|FINNGEN_R12| 2303|
|       GCST|94101|
+-----------+-----+



In [8]:
mapping=pd.read_csv("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/Nelsons_grouping_EFO.tsv",sep="\t")

In [9]:
mapping

,Nelson's grouping name,Mapped MeSH from the paper,List of EFO links,EFOs
0,Haematology,C15,http://www.ebi.ac.uk/efo/EFO_0005803\n http://...,EFO_0005803;EFO_0007352
1,Metabolic,C18,http://www.ebi.ac.uk/efo/EFO_0000589,EFO_0000589
2,Congenital,C16,https://platform.opentargets.org/disease/OTAR_...,OTAR_0000018
3,Signs/symptoms,C23,https://platform.opentargets.org/disease/EFO_0...,EFO_0003765
4,Neurology,C10,https://platform.opentargets.org/disease/EFO_0...,EFO_0000618;HP_0000707
5,Immune,C20\nG12,http://www.ebi.ac.uk/efo/EFO_0000540,EFO_0000540
6,Psychiatry,F03\n F01\n F02,https://platform.opentargets.org/disease/MONDO...,MONDO_0002025
7,Dermatology,C17\n G13,https://platform.opentargets.org/disease/EFO_0...,EFO_0010285
8,Ophthalmology,C11,https://platform.opentargets.org/disease/MONDO...,MONDO_0024458;HP_0000478
9,Cardiovascular,C14\n A07,https://platform.opentargets.org/disease/EFO_0...,EFO_0000319;HP_0001626


In [10]:
disease_index_path=path_to_release_folder+"output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

In [11]:
efo_to_assign=chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
        disease_index_orig=disease_index_orig, efo_ids=["OTAR_0000018"])
len(efo_to_assign)

10498

In [12]:
si.printSchema()

root
 |-- studyId: string (nullable = true)
 |-- projectId: string (nullable = true)
 |-- studyType: string (nullable = true)
 |-- traitFromSource: string (nullable = true)
 |-- traitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- diseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- geneId: string (nullable = true)
 |-- biosampleFromSourceId: string (nullable = true)
 |-- biosampleId: string (nullable = true)
 |-- pubmedId: string (nullable = true)
 |-- publicationTitle: string (nullable = true)
 |-- publicationFirstAuthor: string (nullable = true)
 |-- publicationDate: string (nullable = true)
 |-- publicationJournal: string (nullable = true)
 |-- backgroundTraitFromSourceMappedIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- backgroundDiseaseIds: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- initialSampleSize: string (nullable = true)
 

In [13]:
for index, row in mapping.iterrows():
    efo_list = row["EFOs"].split(";")
    col_name= row["Nelson's grouping name"]

    efo_to_assign=chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
        disease_index_orig=disease_index_orig, efo_ids=efo_list)

    efo_array = f.array(*[f.lit(efo) for efo in efo_to_assign])

    si = si.withColumn(
        f"{col_name}",
        f.when(
            f.size(f.array_intersect(f.col("diseaseIds"), efo_array)) > 0,
            1
        ).otherwise(0)
    )

    print(f"Row {col_name}: {efo_list} : {len(efo_to_assign)}")

Row Haematology: ['EFO_0005803', 'EFO_0007352'] : 1094


Row Metabolic: ['EFO_0000589'] : 2078


Row Congenital: ['OTAR_0000018'] : 10498


Row Signs/symptoms: ['EFO_0003765'] : 94


Row Neurology: ['EFO_0000618', 'HP_0000707'] : 4481


...

Row Respiratory: ['EFO_0000684'] : 566


Row Digestive: ['EFO_0010282'] : 1308


Row Endocrine: ['EFO_0001379', 'HP_0000818'] : 1446


Row Musculoskeletal: ['HP_0000924', 'EFO_0009676'] : 3144


Row Infection: ['EFO_0000544', 'EFO_0000771', 'EFO_0000763', 'EFO_0001067'] : 643


In [15]:
si.show(5,truncate=False)

+--------------------------------------+-----------+---------+----------------------------------------------------+------------------------+---------------+------+---------------------+-----------+--------+----------------------------------------------------------------------------------------------------+----------------------+---------------+------------------+----------------------------------+--------------------+-------------------------------------------+------+---------+--------+---------+---------------------+-------------------+------------------+----------------------------------------------------------+-------------+------------------------------------------------------------------------------------+-----------+---------+---------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+
|studyId                               |projectId  |study

In [16]:
si.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/tmp_v1")

25/04/03 16:43:38 WARN DAGScheduler: Broadcasting large task binary with size 1196.8 KiB


In [17]:
si_mod=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/tmp_v1")

In [21]:

si_mod.count()

96404

In [18]:
efo_to_assign=chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig, efo_ids=["EFO_0001444"])
len(efo_to_assign)

8293

In [19]:
efo_array = f.array(*[f.lit(efo) for efo in efo_to_assign])

si_mod = si_mod.withColumn(
    "Measurement",
    f.when(
        f.size(f.array_intersect(f.col("diseaseIds"), efo_array)) > 0,
        1
    ).otherwise(0)
).cache()

In [20]:
si_mod.show(5)

+--------------------+-----------+---------+--------------------+------------------------+---------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+-------------------+------------------+--------------------+-------------+--------------------+-----------+---------+---------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|     diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMappedIds|backgroundDiseaseIds|   initia

In [22]:
si_mod = si_mod.withColumn(
    "bianry_less_cases",
    f.when(
        f.col("nCases") < f.col("nControls"),
        1
    ).otherwise(0)
)

In [23]:
si_mod.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMapp

In [24]:
si_mod.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/tmp_v2")

# Patching

In [25]:
si_mod_2=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/tmp_v2")

In [ ]:
si_mod_2.count()

96404

In [27]:
si_mod_2.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSourceMapp

In [28]:
# List of columns to update
columns_to_update = [
    "Haematology", "Metabolic", "Congenital", "Signs/symptoms", "Neurology",
    "Immune", "Psychiatry", "Dermatology", "Ophthalmology", "Cardiovascular",
    "Respiratory", "Digestive", "Endocrine", "Musculoskeletal", "Infection"
]

columns_to_update_oncology = [
    "Haematology", "Metabolic", "Congenital", "Signs/symptoms", "Neurology",
    "Immune", "Psychiatry", "Dermatology", "Ophthalmology", "Cardiovascular",
    "Respiratory", "Digestive", "Endocrine", "Musculoskeletal", "Infection","Oncology"
]

In [29]:
# Step 1: Assign 0 if Measurement == 1 or bianry_less_cases == 0
for col in columns_to_update_oncology:
    si_mod_2 = si_mod_2.withColumn(
        col,
        f.when(
            (f.col("Measurement") == 1) | (f.col("bianry_less_cases") == 0),
            0
        ).otherwise(f.col(col))
    )



In [30]:
# Step 2: Assign 0 if Oncology == 1
for col in columns_to_update:
    si_mod_2 = si_mod_2.withColumn(
        col,
        f.when(
            f.col("Oncology") == 1,
            0
        ).otherwise(f.col(col))
    )


In [31]:
# Step 3: Create the "Other" column
si_mod_2 = si_mod_2.withColumn(
    "Other",
    f.when(
        (f.col("Measurement") == 0) &
        (f.col("bianry_less_cases") == 1) &
        (sum(f.col(col) for col in columns_to_update_oncology) == 0),
        1
    ).otherwise(0)
)

In [32]:
si_mod_2.show()

+--------------------+-----------+---------+--------------------+------------------------+--------------------+------+---------------------+-----------+--------+--------------------+----------------------+---------------+------------------+----------------------------------+--------------------+--------------------+------+---------+--------+---------+---------------------+--------------------+--------------------+--------------------+-------------+--------------------+-----------+---------+--------------------+-----------+---------+----------+--------------+---------+------+----------+-----------+-------------+--------------+--------+-----------+---------+---------+---------------+---------+-----------+-----------------+-----+
|             studyId|  projectId|studyType|     traitFromSource|traitFromSourceMappedIds|          diseaseIds|geneId|biosampleFromSourceId|biosampleId|pubmedId|    publicationTitle|publicationFirstAuthor|publicationDate|publicationJournal|backgroundTraitFromSour

In [34]:
# Create a dictionary to store the counts
study_counts = {}

# Loop through each column and count the number of studies where the column equals 1
for col in columns_to_update_oncology + ["Other"]+["Measurement"]:
    count = si_mod_2.filter(f.col(col) == 1).count()
    study_counts[col] = count

# Output the counts
for col, count in study_counts.items():
    print(f"{col}: {count} studies")

Haematology: 199 studies
Metabolic: 687 studies
Congenital: 1193 studies
Signs/symptoms: 392 studies
Neurology: 2686 studies
Immune: 1177 studies
Psychiatry: 1200 studies
Dermatology: 721 studies
Ophthalmology: 847 studies
Cardiovascular: 1698 studies
Respiratory: 811 studies
Digestive: 1244 studies
Endocrine: 804 studies
Musculoskeletal: 1780 studies
Infection: 667 studies
Oncology: 2227 studies
Other: 2983 studies
Measurement: 77948 studies


In [35]:
si_mod_2.write.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/gwas_study_index_with_theraputic_areas")

# Playing with the data

In [9]:
df=session.spark.read.parquet("gs://genetics-portal-dev-analysis/yt4/20250403_for_gentropy_paper/gwas_study_index_with_theraputic_areas")

In [10]:
columns_to_update_oncology = [
    "Haematology", "Metabolic", "Congenital", "Signs/symptoms", "Neurology",
    "Immune", "Psychiatry", "Dermatology", "Ophthalmology", "Cardiovascular",
    "Respiratory", "Digestive", "Endocrine", "Musculoskeletal", "Infection","Oncology"
]

In [11]:
# Create a dictionary to store the counts
study_counts = {}

# Loop through each column and count the number of studies where the column equals 1
for col in columns_to_update_oncology + ["Other"]+["Measurement"]:
    count = df.filter(f.col(col) == 1).count()
    study_counts[col] = count

# Output the counts
for col, count in study_counts.items():
    print(f"{col}: {count} studies")

Haematology: 199 studies
Metabolic: 687 studies
Congenital: 1193 studies
Signs/symptoms: 392 studies
Neurology: 2686 studies
Immune: 1177 studies
Psychiatry: 1200 studies
Dermatology: 721 studies
Ophthalmology: 847 studies
Cardiovascular: 1698 studies
Respiratory: 811 studies
Digestive: 1244 studies
Endocrine: 804 studies
Musculoskeletal: 1780 studies
Infection: 667 studies
Oncology: 2227 studies
Other: 2983 studies
Measurement: 77948 studies


In [14]:
study_counts = {}

for col in columns_to_update_oncology + ["Other"]+["Measurement"]:
    max_nSamples = df.filter(f.col(col) == 1).agg(f.max("nSamples").alias("max_nSamples")).collect()[0]["max_nSamples"]
    study_counts[col] = max_nSamples

for col, count in study_counts.items():
    print(f"{col}: {count} max nSamples")

Haematology: 668150 max nSamples
Metabolic: 2206883 max nSamples
Congenital: 2525730 max nSamples
Signs/symptoms: 1028957 max nSamples
Neurology: 2884318 max nSamples
Immune: 2656140 max nSamples
Psychiatry: 2884318 max nSamples
Dermatology: 1618752 max nSamples
Ophthalmology: 1921625 max nSamples
Cardiovascular: 2339188 max nSamples
Respiratory: 2641520 max nSamples
Digestive: 1858897 max nSamples
Endocrine: 2063820 max nSamples
Musculoskeletal: 2656140 max nSamples
Infection: 2974917 max nSamples
Oncology: 2845029 max nSamples
Other: 1003399 max nSamples
Measurement: 5314291 max nSamples


25/04/17 21:57:36 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 994697 ms exceeds timeout 120000 ms
25/04/17 21:57:36 WARN SparkContext: Killing executors is not supported by current scheduler.
25/04/17 22:01:03 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1223)
	at o